# Personalized EEG Sensor Selection for Seizure Prediction

**Research question:** Can seizures be accurately detected using fewer EEG sensors, and how can personalized sensor selection improve detection compared with a one-size-fits-all approach?

This notebook searches for a compact patient-specific montage without brute-forcing all $2^{31}$ sensor combinations. It uses greedy forward selection with swap refinement, grouped nested cross-validation, and an elbow/plateau rule.

## What the algorithm reports

- Selected zero-based sensor indices and channel names
- Optimal number of sensors, $k$
- Event-level sensitivity
- False-alarm episodes per interictal hour
- Mean warning time before seizure onset
- Brier probability-calibration score
- Performance curves versus number of sensors
- Sensor-selection stability across outer CV folds

The inner CV folds select sensors and $k$. Untouched outer folds estimate achieved performance, preventing optimistic results caused by selecting and evaluating sensors on the same data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from seizure_sensor_selection import (
    PersonalizedSensorSelector,
    compare_personalized_with_global,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## 1. Prepare one patient's data

`X_patient` must have shape `(n_windows, 31, n_features_per_channel)`. Features such as bandpower, entropy, line length, or model embeddings must remain separated along the sensor axis.

`groups_patient` must identify independent recordings, sessions, or seizure episodes. This prevents overlapping or correlated windows from crossing CV folds.

For overlapping windows, `duration_hours` should contain the **window stride**, not the full window width.

In [ ]:
# Replace the right-hand sides with your existing model inputs.
#
# X_patient = ...
# y_patient = ...                    # 1=preictal, 0=interictal
# groups_patient = ...               # one recording/session ID per window
# channel_names = ...                # 31 names ordered like X_patient[:, :, :]
# window_stride_seconds = ...
# seizure_id_per_window = ...        # unique event ID; -1 when interictal
# minutes_to_onset = ...             # NaN when interictal
# window_start_minutes = ...         # time within each recording
#
# metadata_patient = {
#     "duration_hours": np.full(len(y_patient), window_stride_seconds / 3600),
#     "seizure_ids": seizure_id_per_window,
#     "time_to_seizure_minutes": minutes_to_onset,
#     "window_start_minutes": window_start_minutes,
# }
#
# assert X_patient.ndim == 3
# assert X_patient.shape[1] == 31
# assert len(y_patient) == len(groups_patient) == len(X_patient)

## 2. Configure the selector

Replace `estimator=None` with your existing probability-producing model. It must implement `fit(X, y)` and `predict_proba(X)`. Leaving it as `None` uses a class-balanced logistic-regression baseline.

The default utility keeps sensitivity dominant, penalizes false alarms and Brier score, and lightly rewards earlier warnings. Pre-register the utility weights or provide a custom `utility_fn` before final evaluation.

In [ ]:
selector = PersonalizedSensorSelector(
    estimator=None,                 # Replace with your existing model
    max_sensors=31,
    inner_splits=4,
    outer_splits=5,
    threshold=0.5,
    refractory_minutes=30,
    elbow_tolerance=0.02,
    swap_refinement=True,
    random_state=42,
)

## 3. Run personalized sensor selection

Run this block separately for each patient. The saved PNG contains utility, sensitivity, false alarms/hour, warning time, and Brier score versus $k$.

In [ ]:
# Uncomment after loading the patient arrays in Section 1.
#
# result = selector.fit_select(
#     X_patient,
#     y_patient,
#     metadata=metadata_patient,
#     groups=groups_patient,
#     sensor_names=channel_names,
#     plot_path="patient_sensor_selection_curve.png",
# )

## 4. Selected sensors and nested-CV performance

The selected montage is refit using all available patient data. The displayed performance is calculated from outer-fold predictions and is therefore separate from the final refit.

In [ ]:
# selected_sensors = pd.DataFrame({
#     "sensor_index": result.selected_sensor_indices,
#     "sensor_name": result.selected_sensor_names,
# })
# display(selected_sensors)
#
# metrics = result.nested_cv_metrics
# achieved_performance = pd.Series({
#     "optimal_k": result.elbow_k,
#     "sensitivity": metrics.sensitivity,
#     "false_alarms_per_hour": metrics.false_alarms_per_hour,
#     "mean_warning_time_minutes": metrics.mean_warning_time_minutes,
#     "brier_score": metrics.brier_score,
#     "detected_seizures": metrics.n_detected_seizures,
#     "evaluated_seizures": metrics.n_seizures,
#     "false_alarm_episodes": metrics.n_false_alarms,
#     "interictal_hours": metrics.interictal_hours,
# }, name="nested_CV")
# display(achieved_performance.to_frame())
#
# print(result.summary())

## 5. Performance versus number of sensors and elbow

The dashed red line marks the smallest $k$ within `elbow_tolerance` of the best inner-CV utility. This plateau rule is more stable than a purely geometric knee when CV curves are noisy or non-monotonic.

In [ ]:
# display(result.figure)
#
# curve_table = pd.DataFrame([
#     {
#         "k": point.k,
#         "sensor_indices": list(point.sensors),
#         "utility": point.utility_mean,
#         "utility_se": point.utility_se,
#         "sensitivity": point.metrics.sensitivity,
#         "false_alarms_per_hour": point.metrics.false_alarms_per_hour,
#         "mean_warning_time_minutes": point.metrics.mean_warning_time_minutes,
#         "brier_score": point.metrics.brier_score,
#     }
#     for point in result.selection_curve
# ])
# display(curve_table)

## 6. Sensor-selection stability

A sensor appearing in most outer-fold selections is more stable than one selected in only one fold. Report these frequencies with the final montage.

In [ ]:
# selection_count = np.zeros(X_patient.shape[1], dtype=int)
# for fold_subset in result.outer_fold_sensor_indices:
#     selection_count[fold_subset] += 1
#
# stability = (
#     pd.DataFrame({
#         "sensor_index": np.arange(X_patient.shape[1]),
#         "sensor_name": channel_names,
#         "outer_fold_selections": selection_count,
#         "selection_fraction": selection_count / len(result.outer_fold_sensor_indices),
#     })
#     .sort_values(["outer_fold_selections", "sensor_index"], ascending=[False, True])
#     .reset_index(drop=True)
# )
# display(stability)

## 7. Run all patients

Store each patient's arrays in `patient_data`. This loop produces one personalized result and one elbow plot per patient.

In [ ]:
# Expected dictionary structure:
# patient_data = {
#     "PN00": {
#         "X": X_pn00,
#         "y": y_pn00,
#         "metadata": metadata_pn00,
#         "groups": groups_pn00,
#         "sensor_names": channel_names_pn00,
#     },
#     ...
# }
#
# personalized_results = {}
# for patient_id, data in patient_data.items():
#     patient_selector = PersonalizedSensorSelector(
#         estimator=None,  # Use your model here
#         max_sensors=31,
#         inner_splits=4,
#         outer_splits=5,
#         random_state=42,
#     )
#     personalized_results[patient_id] = patient_selector.fit_select(
#         data["X"],
#         data["y"],
#         metadata=data["metadata"],
#         groups=data["groups"],
#         sensor_names=data["sensor_names"],
#         plot_path=f"results/{patient_id}_sensor_curve.png",
#     )

## 8. Personalized versus one-size-fits-all

For the global condition, concatenate patients and group CV by a unique compound patient/recording ID. The global montage must be selected inside its own nested CV rather than selected once before evaluation.

Compare macro-averaged patient metrics so patients with long recordings do not dominate the result.

In [ ]:
# After fitting both personalized_results and global_result:
#
# comparison = compare_personalized_with_global(
#     personalized_results,
#     global_result,
# )
# display(pd.Series(comparison, name="value").to_frame())

## Reporting notes

- **Sensitivity:** proportion of seizure events with at least one preictal alarm.
- **False alarms/hour:** distinct interictal alarm episodes divided by interictal monitoring hours; alarms inside the refractory interval count once.
- **Mean warning time:** mean interval from the earliest correct alarm to seizure onset among detected events.
- **Brier score:** mean squared error of held-out seizure probabilities; lower is better.
- Report fold-level confidence intervals and selection frequencies in addition to pooled estimates.
- Patients with too few independent recordings for the requested fold count need fewer folds or a leave-one-seizure/recording-out design.
- Do not choose utility weights, thresholds, or the elbow tolerance after examining outer-fold results.